# M21 · Diffusion & visual generation

_Curriculum · Domain 4 · GenAI_

We build a tiny CPU-only diffusion demo. The forward formula is $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$, and the reverse process will denoise a toy distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(21)

## A toy creative distribution

Instead of images, we use 2-D points from two clusters. Think of the axes as two learned visual features, such as warmth and contrast.

In [ ]:
n = 600
left = rng.normal(loc=[-2.0, 0.0], scale=0.35, size=(n // 2, 2))
right = rng.normal(loc=[2.0, 0.0], scale=0.35, size=(n // 2, 2))
x0 = np.vstack([left, right])
labels = np.array([0] * (n // 2) + [1] * (n // 2))

print(x0.shape)
assert x0.shape == (600, 2)

## Noise schedule

A schedule chooses $\beta_t$, then $\alpha_t=1-\beta_t$, then $\bar\alpha_t=\prod_{s=1}^t\alpha_s$. Smaller $\bar\alpha_t$ means less original signal remains.

In [ ]:
timesteps = 60
beta = np.linspace(0.0005, 0.05, timesteps)
alpha = 1.0 - beta
alpha_bar = np.cumprod(alpha)

print(alpha_bar[:5].round(4))
print(alpha_bar[-1].round(4))
assert np.all(np.diff(alpha_bar) < 0)

## Forward diffusion

We can sample any timestep directly from the closed form. That is why diffusion training can pick random timesteps instead of simulating every earlier step.

In [ ]:
def q_sample(clean, step, noise):
    signal = np.sqrt(alpha_bar[step]) * clean
    scaled_noise = np.sqrt(1.0 - alpha_bar[step]) * noise
    return signal + scaled_noise

noise = rng.normal(size=x0.shape)
x_mid = q_sample(x0, 25, noise)
x_late = q_sample(x0, 59, noise)

print(np.var(x0).round(3))
print(np.var(x_late).round(3))
assert x_mid.shape == x0.shape

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
sets = [(x0, "clean"), (x_mid, "mid noise"), (x_late, "late noise")]
for ax, item in zip(axes, sets):
    points = item[0]
    title = item[1]
    ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.55)
    ax.set_title(title)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-3, 3)
plt.show()

## A tiny reverse denoiser

For the toy distribution, we know the two cluster centers. A simple denoiser can pull each point toward the nearest center, with stronger pulls as the sample gets cleaner.

In [ ]:
centers = np.array([[-2.0, 0.0], [2.0, 0.0]])

def nearest_center(points):
    distances = ((points[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    closest = distances.argmin(axis=1)
    return centers[closest]

def reverse_step(points, step):
    target = nearest_center(points)
    strength = 0.08 + 0.20 * (1.0 - alpha_bar[step])
    fresh_noise = rng.normal(scale=0.025, size=points.shape)
    updated = points + strength * (target - points) + fresh_noise
    return updated

In [ ]:
sample = rng.normal(size=(600, 2))
trajectory = [sample.copy()]
for step in range(timesteps - 1, -1, -1):
    sample = reverse_step(sample, step)
    if step in [45, 30, 15, 0]:
        trajectory.append(sample.copy())

final_mean_abs_y = np.abs(sample[:, 1]).mean()
print(round(final_mean_abs_y, 3))
assert final_mean_abs_y < 0.35

In [ ]:
fig, axes = plt.subplots(1, len(trajectory), figsize=(14, 3))
titles = ["noise", "t=45", "t=30", "t=15", "final"]
for ax, points, title in zip(axes, trajectory, titles):
    ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.55)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-3, 3)
    ax.set_title(title)
plt.show()

## Guidance in one dimension

Classifier-free guidance combines unconditional and conditional noise predictions: $\hat\epsilon=\epsilon_{uncond}+w(\epsilon_{cond}-\epsilon_{uncond})$.

In [ ]:
eps_uncond = 0.25
eps_cond = -0.10
w = 2.5
eps_guided = eps_uncond + w * (eps_cond - eps_uncond)

print(eps_guided)
assert eps_guided < eps_cond

## Takeaway

A production visual model replaces our nearest-center rule with a neural denoiser, and replaces the toy condition with text, image, or brand constraints. The schedule, forward noising, reverse denoising, and guidance idea stay the same.